# 板内图案演示（不需要 HDMI 输入）

运行这一格之后，**采集卡窗口里应该出现彩条 + 灰阶 + 一个白块**。
如果看不到画面，先确认：采集卡 USB 插好、相机/OBS 选的是采集卡、板子 HDMI OUT 接到采集卡。


In [ ]:
from pynq import Overlay
from pynq.lib.video import VideoMode
import pattern_demo as pd

ol = Overlay("enhance_pattern.bit")
video = ol.video
video.hdmi_out.configure(VideoMode(1280, 720, 24, 60))
cfg = pd.find(ol, "gpio_enh_cfg")
lat = pd.find(ol, "gpio_enh_lat")
print("overlay loaded, hdmi_out = 1280x720@60")
print("capture card should now show colour bars")


## 换一个预设（画面立刻变）

改下面 `PRESET` 的值再运行：`pass` / `clarity` / `reading` / `document` / `night` /
`invert` / `binarize` / `split`。


In [ ]:
PRESET = "document"
lo, hi = pd.pack(pd.PRESETS[PRESET])
cfg.mmio.write(0x00, lo)
cfg.mmio.write(0x08, hi)
a, b = lat.mmio.read(0x00), lat.mmio.read(0x08)
n = (b - a) & 0xFFFFFFFF
print(f"preset {PRESET} applied; latency {n} pixel clocks (~{n / 142.857:.3f} us)")


## 自动轮播（录视频用这一格）

每 `HOLD` 秒换一个预设，采集卡画面跟着变。想停就点工具栏的 ⏹。


In [ ]:
import time

HOLD = 4
ORDER = ["pass", "clarity", "document", "night", "invert", "binarize", "split"]
for name in ORDER:
    lo, hi = pd.pack(pd.PRESETS[name])
    cfg.mmio.write(0x00, lo)
    cfg.mmio.write(0x08, hi)
    a, b = lat.mmio.read(0x00), lat.mmio.read(0x08)
    n = (b - a) & 0xFFFFFFFF
    print(f"[{time.strftime('%H:%M:%S')}] {name:<9} latency {n} clocks (~{n / 142.857:.3f} us)")
    time.sleep(HOLD)
print("done")
